# Data · 시계열 원본 빈칸 7문제

문제 원본은 **ts_practice.ipynb**, 정답 기준은 **ts_solution.ipynb**입니다. copy 파일은 정답 근거로 사용하지 않습니다. 원본의 미완성5개 셀을 학습/평가, 지표, Conv1D, RNN, Encoder/Decoder의7문제로 묶었습니다. 추가로 만든 문제는 제외했습니다.

**문제 → 빈칸 코드 → 검증 코드 → 접힌 해설·정답** 순서입니다. 첫 준비 셀을 실행하고 MISSING 또는 NotImplementedError 부분을 완성하세요. 문제 셀→검증 셀 순서로 실행합니다. SyntaxError가 나면 수정 후 문제 셀부터 재실행하세요.

CPU PyTorch, NumPy, scikit-learn만 필요합니다. 주가 데이터·인터넷·GPU 없이 작은 더미 시계열로 검사합니다. 긴10epoch 학습·그래프는 제외했습니다. PASS는 작은 입력의 검증 결과이며 실제 예측 품질·숙련을 뜻하지 않습니다.

Decoder의 FC는 정답 파일과 같은 nn.Linear(hidden_size,input_size)입니다. 원본 파일은 수정하지 않았습니다.

In [ ]:
import math
import copy
import numpy as np
import torch
import torch.nn as nn  # nn은 torch.nn의 별명입니다.
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
torch.set_num_threads(1)
MISSING = object()  # ???? 옆의 MISSING을 정답 코드로 교체하세요.

class TinyRegressor(nn.Module):
    def __init__(self, features=2):
        super().__init__()
        self.fc=nn.Linear(features,1)
        self.calls=[]
    def forward(self,x):
        self.calls.append((self.training,torch.is_grad_enabled()))
        return self.fc(x)

def snapshot(x):
    if isinstance(x,torch.Tensor):return x.detach().clone()
    if isinstance(x,np.ndarray):return x.copy()
    if isinstance(x,dict):return {k:snapshot(v) for k,v in x.items()}
    if isinstance(x,(tuple,list)):return tuple(snapshot(v) for v in x)
    return x

def same(a,e):
    if isinstance(e,torch.Tensor):torch.testing.assert_close(a,e,rtol=2e-5,atol=2e-6)
    elif isinstance(e,np.ndarray):np.testing.assert_allclose(a,e,rtol=1e-6,atol=1e-7)
    elif isinstance(e,dict):
        assert a.keys()==e.keys()
        for k in e:same(a[k],e[k])
    elif isinstance(e,tuple):
        assert len(a)==len(e)
        for v,w in zip(a,e):same(v,w)
    elif isinstance(e,(float,np.floating)):assert abs(a-e)<1e-6,(a,e)
    else:assert a==e,(a,e)


class EncoderRNN(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers):
    super(EncoderRNN, self).__init__()
    self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

  def forward(self, x):
    _, h = self.rnn(x)
    return h

class DecoderRNN(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers):
    super(DecoderRNN, self).__init__()
    self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
    self.fc = nn.Linear(hidden_size, input_size)

  def forward(self, x, h):
    out, h = self.rnn(x, h)
    out = self.fc(out)
    return out, h

class RNNRNN(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers):
    super(RNNRNN, self).__init__()
    self.encoder = EncoderRNN(input_size, hidden_size, num_layers)
    self.decoder = DecoderRNN(input_size, hidden_size, num_layers)

  def forward(self, source, target_len):
    h = self.encoder(source)
    predictions = []
    input = source[:, -1, :].unsqueeze(1)
    for t in range(target_len):
      out, h = self.decoder(input, h)
      predictions.append(out.squeeze(1))
      input = out
    outputs = torch.stack(predictions, dim=1)

    return outputs

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.linear(out)

def observe(actual,kind,case,ctx):
    if kind=='eval':
        losses,calls=actual
        assert all(not training and not grad for training,grad in calls)
        assert all(not v.requires_grad for v in losses)
    if kind in ['Conv1DModel','RNNModel','EncoderRNN','DecoderRNN','LSTMModel','RNNRNN']:
        m=actual
        x=torch.randn(2,5,ctx['input_size'],requires_grad=True)
        if kind=='DecoderRNN':
            h=torch.randn(ctx['num_layers'],2,ctx['hidden_size'],requires_grad=True)
            y=m(x,h)
            assert y[0].shape==(2,5,ctx['input_size'])
        elif kind=='RNNRNN':
            y=m(x,case+2)
            assert y.shape==(2,case+2,ctx['input_size'])
        else:y=m(x)
        if kind=='EncoderRNN':assert y.shape==(ctx['num_layers'],2,ctx['hidden_size'])
        if kind in ['RNNModel','LSTMModel']:assert y.shape==(2,5,1)
        if kind=='Conv1DModel':assert y.shape==(2,4,1)
        loss=sum(v.square().mean() for v in y) if isinstance(y,tuple) else y.square().mean()
        loss.backward()
        assert x.grad is not None and x.grad.abs().sum()>0
        grads=tuple((n,p.grad) for n,p in m.named_parameters())
        return snapshot((y,grads))
    return snapshot(actual)


## 01. 한 시점 예측: 학습 배치

**구분: 원본 빈칸**. 원본 전체 셀 **24** (Markdown 포함,1부터). 문제는 ts_practice.ipynb, 정답은 ts_solution.ipynb를 기준으로 합니다.

목표: 제공 문맥에서 빈칸을 구현하세요. 반환값은 `(losses,model.state_dict(),model.calls)`입니다.

**입력 예시:**
```python
case=0
model=TinyRegressor()
model.eval()
optimizer=torch.optim.Adam(model.parameters(),lr=.01+case*.01)
x=torch.arange(24).float().reshape(2,6,2)/10
batches=[(x,torch.tensor([1.,2.])),(x.flip(1),torch.tensor([2.,1.]))]
loss_fn=nn.MSELoss()
device='cpu'
```

In [ ]:
import torch
import torch.nn as nn

def solve_01(model, batches, optimizer, loss_fn, device):
    model.train()
    losses=[]
    for batch_x,batch_y in batches:
        batch_x,batch_y=MISSING  # ???? 빈칸
        pred=MISSING  # ???? 빈칸
        loss=MISSING  # ???? 빈칸
        optimizer.MISSING()  # ???? 빈칸
        loss.MISSING()  # ???? 빈칸
        optimizer.MISSING()  # ???? 빈칸
        losses.append(loss.detach())
    return (losses,model.state_dict(),model.calls)

In [ ]:
# 이 검증 셀은 수정하지 마세요.
def check_01():
    expected=(((torch.tensor(13.172225952148438,dtype=torch.float32), torch.tensor(4.5301103591918945,dtype=torch.float32),), {'fc.weight':torch.tensor([[-0.6098431348800659, -0.6203716397285461]],dtype=torch.float32), 'fc.bias':torch.tensor([0.20660428702831268],dtype=torch.float32)}, ((True, True,), (True, True,),),), ((torch.tensor(1.206669569015503,dtype=torch.float32), torch.tensor(2.8444483280181885,dtype=torch.float32),), {'fc.weight':torch.tensor([[-0.11213381588459015, 0.642203152179718]],dtype=torch.float32), 'fc.bias':torch.tensor([-0.32645532488822937],dtype=torch.float32)}, ((True, True,), (True, True,),),),)
    for case in range(2):
        torch.manual_seed(3010+case)
        model=TinyRegressor()
        model.eval()
        optimizer=torch.optim.Adam(model.parameters(),lr=.01+case*.01)
        x=torch.arange(24).float().reshape(2,6,2)/10
        batches=[(x,torch.tensor([1.,2.])),(x.flip(1),torch.tensor([2.,1.]))]
        loss_fn=nn.MSELoss()
        device='cpu'
        actual=solve_01(model, batches, optimizer, loss_fn, device)
        same(observe(actual,'value',case,locals()),expected[case])
    print('PASS · 01 · 두 입력 검증')
try:
    check_01()
except Exception as error:
    print('FAIL · 01 ·',type(error).__name__,str(error))


<details><summary>해설 · 공식 · 정답 보기 (먼저 풀어보세요)</summary>

출력(B,T,1)에서 [:,-1,0]→(B,). MSE→zero_grad→backward→step. 두 배치를 실행해 gradient 누적을 검사합니다. 긴10epoch와 그래프는 제외했습니다.

```python
def solve_01(model, batches, optimizer, loss_fn, device):
    model.train()
    losses=[]
    for batch_x,batch_y in batches:
        batch_x,batch_y=batch_x.to(device),batch_y.to(device)
        pred=model(batch_x)[:, -1, 0]
        loss=loss_fn(pred,batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.detach())
    return (losses,model.state_dict(),model.calls)
```

테스트용 함수 인자·반환값을 추가하고 긴 실행·다운로드·그래프는 제외했습니다.
</details>

## 02. 한 시점 예측: 평가 배치

**구분: 원본 빈칸**. 원본 전체 셀 **24** (Markdown 포함,1부터). 문제는 ts_practice.ipynb, 정답은 ts_solution.ipynb를 기준으로 합니다.

목표: 제공 문맥에서 빈칸을 구현하세요. 반환값은 `(losses, model.calls)`입니다.

**입력 예시:**
```python
case=0
model=TinyRegressor()
x=torch.randn(2,4+case,2)
batches=[(x,torch.tensor([1.,2.]))]
loss_fn=nn.MSELoss()
device='cpu'
```

In [ ]:
import torch
import torch.nn as nn

def solve_02(model, batches, loss_fn, device):
    model.eval()
    losses=[]
    with torch.no_grad():
        for test_x,test_y in batches:
            test_x,test_y=MISSING  # ???? 빈칸
            test_pred=MISSING  # ???? 빈칸
            test_loss=MISSING  # ???? 빈칸
            losses.append(test_loss)
    return (losses, model.calls)

In [ ]:
# 이 검증 셀은 수정하지 마세요.
def check_02():
    expected=(((torch.tensor(2.9661526679992676,dtype=torch.float32),), ((False, False,),),), ((torch.tensor(1.490590214729309,dtype=torch.float32),), ((False, False,),),),)
    for case in range(2):
        torch.manual_seed(3020+case)
        model=TinyRegressor()
        x=torch.randn(2,4+case,2)
        batches=[(x,torch.tensor([1.,2.]))]
        loss_fn=nn.MSELoss()
        device='cpu'
        actual=solve_02(model, batches, loss_fn, device)
        same(observe(actual,'eval',case,locals()),expected[case])
    print('PASS · 02 · 두 입력 검증')
try:
    check_02()
except Exception as error:
    print('FAIL · 02 ·',type(error).__name__,str(error))


<details><summary>해설 · 공식 · 정답 보기 (먼저 풀어보세요)</summary>

eval은 Dropout/BN 동작을 바꾸고 no_grad는 gradient 기록을 끕니다. 예: (2,4,1)→마지막 예측(2,).

```python
def solve_02(model, batches, loss_fn, device):
    model.eval()
    losses=[]
    with torch.no_grad():
        for test_x,test_y in batches:
            test_x,test_y=test_x.to(device),test_y.to(device)
            test_pred=model(test_x)[:, -1, 0]
            test_loss=loss_fn(test_pred,test_y)
            losses.append(test_loss)
    return (losses, model.calls)
```

테스트용 함수 인자·반환값을 추가하고 긴 실행·다운로드·그래프는 제외했습니다.
</details>

## 03. RMSE와 MAPE

**구분: 원본 빈칸**. 원본 전체 셀 **28** (Markdown 포함,1부터). 문제는 ts_practice.ipynb, 정답은 ts_solution.ipynb를 기준으로 합니다.

목표: 제공 문맥에서 빈칸을 구현하세요. 반환값은 `(rmse,mape)`입니다.

**입력 예시:**
```python
case=0
y_test=np.array([2.,4.,8.])*(case+1)
test_predictions=np.array([1.,5.,6.])*(case+1)
```

In [ ]:
import torch
import torch.nn as nn

def solve_03(y_test, test_predictions):
    rmse=np.sqrt(mean_squared_error(MISSING, MISSING))  # ???? 빈칸
    mape=mean_absolute_percentage_error(MISSING, MISSING)  # ???? 빈칸
    return (rmse,mape)

In [ ]:
# 이 검증 셀은 수정하지 마세요.
def check_03():
    expected=((1.4142135623730951, 0.3333333333333333,), (2.8284271247461903, 0.3333333333333333,),)
    for case in range(2):
        torch.manual_seed(3030+case)
        y_test=np.array([2.,4.,8.])*(case+1)
        test_predictions=np.array([1.,5.,6.])*(case+1)
        actual=solve_03(y_test, test_predictions)
        same(observe(actual,'value',case,locals()),expected[case])
    print('PASS · 03 · 두 입력 검증')
try:
    check_03()
except Exception as error:
    print('FAIL · 03 ·',type(error).__name__,str(error))


<details><summary>해설 · 공식 · 정답 보기 (먼저 풀어보세요)</summary>

RMSE=sqrt(mean((y-pred)^2)). MAPE=mean(abs(y-pred)/abs(y)), sklearn은 비율로 반환합니다. .25는25%. MAPE에서 y_true가 첫 인자여야 합니다. 정답0인 입력은 이 테스트에서 제외합니다.

```python
def solve_03(y_test, test_predictions):
    rmse=np.sqrt(mean_squared_error(y_test,test_predictions))
    mape=mean_absolute_percentage_error(y_test,test_predictions)
    return (rmse,mape)
```

테스트용 함수 인자·반환값을 추가하고 긴 실행·다운로드·그래프는 제외했습니다.
</details>

## 04. Conv1DModel 구성과 forward

**구분: 원본 빈칸**. 원본 전체 셀 **30** (Markdown 포함,1부터). 문제는 ts_practice.ipynb, 정답은 ts_solution.ipynb를 기준으로 합니다.

목표: 제공 문맥에서 빈칸을 구현하세요. 반환값은 `Conv1DModel(input_size,hidden_size)`입니다.

**입력 예시:**
```python
case=0
input_size=case+1
hidden_size=3+case
```

In [ ]:
import torch
import torch.nn as nn

def solve_04(input_size, hidden_size):
    class Conv1DModel(nn.Module):
      def __init__(self, input_size, hidden_size):
        super(Conv1DModel, self).__init__()
        self.conv1d = nn.Conv1d(in_channels=MISSING, \
                                out_channels=MISSING, \
                                kernel_size=2, stride=1)
        self.fc = nn.Linear(MISSING, 1)  # ???? 빈칸

      def forward(self, x):
        x = x.transpose(1, 2)
        x = self.conv1d(x)
        x = x.transpose(1, 2)
        return self.fc(x)
    return Conv1DModel(input_size,hidden_size)

In [ ]:
# 이 검증 셀은 수정하지 마세요.
def check_04():
    expected=((torch.tensor([[[-0.3281039297580719], [-0.5659602284431458], [-0.3204578161239624], [-0.5607237815856934]], [[-1.0511831045150757], [-0.7583720088005066], [-0.1504526287317276], [-0.4922559857368469]]],dtype=torch.float32), (('conv1d.weight', torch.tensor([[[0.02724059857428074, 0.09344536066055298]], [[-0.04640699550509453, -0.1591932475566864]], [[0.08955425024032593, 0.30720433592796326]]],dtype=torch.float32),), ('conv1d.bias', torch.tensor([0.09100064635276794, -0.15502843260765076, 0.299167275428772],dtype=torch.float32),), ('fc.weight', torch.tensor([[-0.31638795137405396, 0.5063126087188721, -0.9983234405517578]],dtype=torch.float32),), ('fc.bias', torch.tensor([-1.0568773746490479],dtype=torch.float32),),),), (torch.tensor([[[-0.6895064115524292], [-0.5858267545700073], [-0.6187149286270142], [-0.49611008167266846]], [[-0.41493701934814453], [-0.19826853275299072], [-0.19306588172912598], [-0.3551730215549469]]],dtype=torch.float32), (('conv1d.weight', torch.tensor([[[0.05489474534988403, 0.05949699506163597], [-0.10752319544553757, -0.05506923422217369]], [[0.0912022814154625, 0.09884849935770035], [-0.17863933742046356, -0.09149220585823059]], [[0.07099898159503937, 0.07695138454437256], [-0.13906681537628174, -0.07122466713190079]], [[-0.1288524866104126, -0.1396551877260208], [0.25238537788391113, 0.1292620599269867]]],dtype=torch.float32),), ('conv1d.bias', torch.tensor([-0.14823003113269806, -0.2462698370218277, -0.19171564280986786, 0.3479350805282593],dtype=torch.float32),), ('fc.weight', torch.tensor([[0.021839629858732224, -0.1419694423675537, 0.5901132225990295, -0.19539904594421387]],dtype=torch.float32),), ('fc.bias', torch.tensor([-0.8879006505012512],dtype=torch.float32),),),),)
    for case in range(2):
        torch.manual_seed(3040+case)
        input_size=case+1
        hidden_size=3+case
        actual=solve_04(input_size, hidden_size)
        same(observe(actual,'Conv1DModel',case,locals()),expected[case])
    print('PASS · 04 · 두 입력 검증')
try:
    check_04()
except Exception as error:
    print('FAIL · 04 ·',type(error).__name__,str(error))


<details><summary>해설 · 공식 · 정답 보기 (먼저 풀어보세요)</summary>

Conv1d 입력은(B,F,T). 원본(B,T,F)을 transpose한 뒤 처리합니다. kernel2,stride1,padding0이므로 길이T→T-1. 다시 transpose 후 Linear(H,1).

```python
def solve_04(input_size, hidden_size):
    class Conv1DModel(nn.Module):
      def __init__(self, input_size, hidden_size):
        super(Conv1DModel, self).__init__()
        self.conv1d = nn.Conv1d(in_channels=input_size, \
                                out_channels=hidden_size, \
                                kernel_size=2, stride=1)
        self.fc = nn.Linear(hidden_size, 1)

      def forward(self, x):
        x = x.transpose(1, 2)
        x = self.conv1d(x)
        x = x.transpose(1, 2)
        return self.fc(x)
    return Conv1DModel(input_size,hidden_size)
```

테스트용 함수 인자·반환값을 추가하고 긴 실행·다운로드·그래프는 제외했습니다.
</details>

## 05. RNNModel 구성과 forward

**구분: 원본 빈칸**. 원본 전체 셀 **35** (Markdown 포함,1부터). 문제는 ts_practice.ipynb, 정답은 ts_solution.ipynb를 기준으로 합니다.

목표: 제공 문맥에서 빈칸을 구현하세요. 반환값은 `RNNModel(input_size,hidden_size,num_layers)`입니다.

**입력 예시:**
```python
case=0
input_size=case+1
hidden_size=3+case
num_layers=case+1
```

In [ ]:
import torch
import torch.nn as nn

def solve_05(input_size, hidden_size, num_layers):
    class RNNModel(nn.Module):
      def __init__(self, input_size, hidden_size, num_layers):
          super(RNNModel, self).__init__()
          self.rnn = nn.RNN(MISSING, MISSING, MISSING, batch_first=True)  # ???? 빈칸
          self.fc = nn.Linear(MISSING, 1)  # ???? 빈칸

      def forward(self, x):
          return MISSING  # forward 전체를 완성하세요  # ???? 빈칸
    return RNNModel(input_size,hidden_size,num_layers)

In [ ]:
# 이 검증 셀은 수정하지 마세요.
def check_05():
    expected=((torch.tensor([[[0.34788215160369873], [0.42908501625061035], [0.5703064203262329], [0.598202645778656], [0.6227497458457947]], [[0.3559763431549072], [0.4413849413394928], [0.5335870385169983], [0.5953569412231445], [0.336570143699646]]],dtype=torch.float32), (('rnn.weight_ih_l0', torch.tensor([[-0.0363074466586113], [0.13057459890842438], [0.07118243724107742]],dtype=torch.float32),), ('rnn.weight_hh_l0', torch.tensor([[0.018102936446666718, -0.008825777098536491, -0.013868425041437149], [-0.09149631857872009, 0.057011835277080536, 0.07490473240613937], [-0.1020161360502243, 0.06646817922592163, 0.09305426478385925]],dtype=torch.float32),), ('rnn.bias_ih_l0', torch.tensor([0.058801330626010895, -0.19526341557502747, -0.26071029901504517],dtype=torch.float32),), ('rnn.bias_hh_l0', torch.tensor([0.058801330626010895, -0.19526343047618866, -0.26071029901504517],dtype=torch.float32),), ('fc.weight', torch.tensor([[0.6561518311500549, -0.45071935653686523, -0.6191595792770386]],dtype=torch.float32),), ('fc.bias', torch.tensor([0.9662202596664429],dtype=torch.float32),),),), (torch.tensor([[[-0.8128927946090698], [-0.8785736560821533], [-0.9412858486175537], [-0.8763542175292969], [-0.9134951829910278]], [[-0.8025995492935181], [-0.7733831405639648], [-0.8671001195907593], [-0.9485957026481628], [-0.8712856769561768]]],dtype=torch.float32), (('rnn.weight_ih_l0', torch.tensor([[0.023342519998550415, -0.001992744393646717], [-0.036118678748607635, -0.058325812220573425], [0.099403977394104, 0.1087934672832489], [-0.06296001374721527, -0.024774326011538506]],dtype=torch.float32),), ('rnn.weight_hh_l0', torch.tensor([[-0.0008406667038798332, 0.0004437491297721863, 0.004275675863027573, -0.003789454698562622], [0.028226997703313828, 0.02208433486521244, 0.006527595687657595, -0.021517954766750336], [-0.12153209745883942, -0.09835529327392578, -0.03454110398888588, 0.10519386827945709], [0.06180587410926819, 0.056531891226768494, 0.019717400893568993, -0.05048830807209015]],dtype=torch.float32),), ('rnn.bias_ih_l0', torch.tensor([0.021767590194940567, -0.03608788549900055, 0.32700860500335693, -0.20677056908607483],dtype=torch.float32),), ('rnn.bias_hh_l0', torch.tensor([0.021767597645521164, -0.03608788177371025, 0.32700860500335693, -0.20677056908607483],dtype=torch.float32),), ('rnn.weight_ih_l1', torch.tensor([[0.11232192069292068, 0.09848558902740479, 0.04182112589478493, -0.10586895048618317], [0.3182677924633026, 0.26209694147109985, 0.09741450846195221, -0.28077852725982666], [-0.42474713921546936, -0.39709484577178955, -0.1789744347333908, 0.38828137516975403], [-0.2417127639055252, -0.2061605006456375, -0.08808055520057678, 0.21785561740398407]],dtype=torch.float32),), ('rnn.weight_hh_l1', torch.tensor([[0.15548087656497955, 0.0583312064409256, -0.039916038513183594, -0.08931337296962738], [0.34081611037254333, 0.13165371119976044, -0.08726564049720764, -0.19034822285175323], [-0.49614664912223816, -0.19996246695518494, 0.11215350031852722, 0.2834545969963074], [-0.3182876706123352, -0.12252949923276901, 0.06922232359647751, 0.1805291771888733]],dtype=torch.float32),), ('rnn.bias_ih_l1', torch.tensor([-0.2479543834924698, -0.6347234845161438, 0.8731203079223633, 0.5253304839134216],dtype=torch.float32),), ('rnn.bias_hh_l1', torch.tensor([-0.2479543834924698, -0.634723424911499, 0.8731203079223633, 0.5253304839134216],dtype=torch.float32),), ('fc.weight', torch.tensor([[1.350298523902893, 0.597806990146637, -0.31966108083724976, -0.7961203455924988]],dtype=torch.float32),), ('fc.bias', torch.tensor([-1.737113356590271],dtype=torch.float32),),),),)
    for case in range(2):
        torch.manual_seed(3050+case)
        input_size=case+1
        hidden_size=3+case
        num_layers=case+1
        actual=solve_05(input_size, hidden_size, num_layers)
        same(observe(actual,'RNNModel',case,locals()),expected[case])
    print('PASS · 05 · 두 입력 검증')
try:
    check_05()
except Exception as error:
    print('FAIL · 05 ·',type(error).__name__,str(error))


<details><summary>해설 · 공식 · 정답 보기 (먼저 풀어보세요)</summary>

RNN output=(B,T,H),h_n=(layers,B,H). 모든 시점 output을 Linear(H,1)에 전달해(B,T,1)을 반환합니다. output과 최종 hidden을 혼동하지 마세요.

```python
def solve_05(input_size, hidden_size, num_layers):
    class RNNModel(nn.Module):
      def __init__(self, input_size, hidden_size, num_layers):
          super(RNNModel, self).__init__()
          self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
          self.fc = nn.Linear(hidden_size, 1)

      def forward(self, x):
          out, _ = self.rnn(x)
          return self.fc(out)
    return RNNModel(input_size,hidden_size,num_layers)
```

테스트용 함수 인자·반환값을 추가하고 긴 실행·다운로드·그래프는 제외했습니다.
</details>

## 06. EncoderRNN 구성과 forward

**구분: 원본 빈칸**. 원본 전체 셀 **44** (Markdown 포함,1부터). 문제는 ts_practice.ipynb, 정답은 ts_solution.ipynb를 기준으로 합니다.

목표: 제공 문맥에서 빈칸을 구현하세요. 반환값은 `EncoderRNN(input_size,hidden_size,num_layers)`입니다.

**입력 예시:**
```python
case=0
input_size=case+1
hidden_size=3+case
num_layers=case+1
```

In [ ]:
import torch
import torch.nn as nn

def solve_06(input_size, hidden_size, num_layers):
    class EncoderRNN(nn.Module):
        def __init__(self,input_size,hidden_size,num_layers):
            # 초기화 전체를 완성하세요. super().__init__()도 포함합니다.
            raise NotImplementedError('???? 초기화')
        def forward(self,x):
            raise NotImplementedError('???? forward')
    return EncoderRNN(input_size,hidden_size,num_layers)

In [ ]:
# 이 검증 셀은 수정하지 마세요.
def check_06():
    expected=((torch.tensor([[[0.9806394577026367, 0.3962957262992859, -0.6589806079864502], [0.982463002204895, 0.4506397545337677, -0.6238059997558594]]],dtype=torch.float32), (('rnn.weight_ih_l0', torch.tensor([[-0.02778351493179798], [-0.41887152194976807], [0.4476819932460785]],dtype=torch.float32),), ('rnn.weight_hh_l0', torch.tensor([[0.04325048252940178, -0.018853576853871346, -0.03861917182803154], [0.21185292303562164, -0.15814131498336792, -0.18698561191558838], [-0.24004130065441132, 0.17512845993041992, 0.21197809278964996]],dtype=torch.float32),), ('rnn.bias_ih_l0', torch.tensor([0.04871378093957901, 0.2546918988227844, -0.2865794599056244],dtype=torch.float32),), ('rnn.bias_hh_l0', torch.tensor([0.04871378093957901, 0.25469186902046204, -0.2865794599056244],dtype=torch.float32),),),), (torch.tensor([[[-0.0059487116523087025, 0.06694529205560684, 0.11157110333442688, 0.540124237537384], [0.42949017882347107, -0.7532424926757812, -0.40447497367858887, 0.23635903000831604]], [[0.7959834933280945, -0.38281139731407166, 0.4097580313682556, -0.4386581480503082], [0.8936570286750793, -0.42682743072509766, 0.6877924203872681, -0.6118932962417603]]],dtype=torch.float32), (('rnn.weight_ih_l0', torch.tensor([[-0.07228778302669525, -0.04405701160430908], [0.07793674618005753, 0.11006750911474228], [0.05036001652479172, 0.08923524618148804], [0.018200328573584557, 0.0393042229115963]],dtype=torch.float32),), ('rnn.weight_hh_l0', torch.tensor([[0.02813710831105709, 0.03375058248639107, 0.0952625647187233, 0.08030200749635696], [-0.01719602197408676, -0.004137852229177952, -0.047617554664611816, -0.0429922416806221], [-0.010093817487359047, 0.02882562391459942, 0.0076267309486866, -0.007912038825452328], [-0.0044687483459711075, 0.015571356751024723, 0.006609245203435421, 0.0018867896869778633]],dtype=torch.float32),), ('rnn.bias_ih_l0', torch.tensor([0.179512619972229, -0.08084116876125336, -0.009787935763597488, -0.005324790254235268],dtype=torch.float32),), ('rnn.bias_hh_l0', torch.tensor([0.179512619972229, -0.08084116876125336, -0.00978793390095234, -0.005324789322912693],dtype=torch.float32),), ('rnn.weight_ih_l1', torch.tensor([[0.009471091441810131, -0.01476795319467783, -0.004811032209545374, 0.025139499455690384], [-0.01724681630730629, 0.030355803668498993, 0.02031620219349861, -0.025756843388080597], [0.01990426518023014, -0.02868916094303131, -0.013750188052654266, 0.03449542820453644], [-0.024364810436964035, 0.02644231915473938, -0.006436837837100029, -0.05291000008583069]],dtype=torch.float32),), ('rnn.weight_hh_l1', torch.tensor([[0.050775233656167984, 0.010959312319755554, 0.018767299130558968, -0.03277701884508133], [-0.06323593854904175, -0.008114895783364773, -0.023922353982925415, 0.0386914387345314], [0.07840795814990997, 0.009646294638514519, 0.02933652698993683, -0.0496157631278038], [-0.11194756627082825, -0.014388831332325935, -0.05167464166879654, 0.07458412647247314]],dtype=torch.float32),), ('rnn.bias_ih_l1', torch.tensor([0.057062454521656036, -0.07336170971393585, 0.0902748852968216, -0.1247817724943161],dtype=torch.float32),), ('rnn.bias_hh_l1', torch.tensor([0.057062454521656036, -0.07336171716451645, 0.0902748852968216, -0.12478175759315491],dtype=torch.float32),),),),)
    for case in range(2):
        torch.manual_seed(3060+case)
        input_size=case+1
        hidden_size=3+case
        num_layers=case+1
        actual=solve_06(input_size, hidden_size, num_layers)
        same(observe(actual,'EncoderRNN',case,locals()),expected[case])
    print('PASS · 06 · 두 입력 검증')
try:
    check_06()
except Exception as error:
    print('FAIL · 06 ·',type(error).__name__,str(error))


<details><summary>해설 · 공식 · 정답 보기 (먼저 풀어보세요)</summary>

Encoder는 RNN의 마지막 hidden만 반환합니다. (B,T,F)→(layers,B,H). batch_first=True여도 hidden의 순서는 바뀌지 않습니다.

```python
def solve_06(input_size, hidden_size, num_layers):
    class EncoderRNN(nn.Module):
      def __init__(self, input_size, hidden_size, num_layers):
        super(EncoderRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

      def forward(self, x):
        _, h = self.rnn(x)
        return h
    return EncoderRNN(input_size,hidden_size,num_layers)
```

테스트용 함수 인자·반환값을 추가하고 긴 실행·다운로드·그래프는 제외했습니다.
</details>

## 07. DecoderRNN 구성과 forward

**구분: 원본 빈칸**. 원본 전체 셀 **44** (Markdown 포함,1부터). 문제는 ts_practice.ipynb, 정답은 ts_solution.ipynb를 기준으로 합니다.

목표: 제공 문맥에서 빈칸을 구현하세요. 반환값은 `DecoderRNN(input_size,hidden_size,num_layers)`입니다.

**입력 예시:**
```python
case=0
input_size=case+1
hidden_size=3+case
num_layers=case+1
```

In [ ]:
import torch
import torch.nn as nn

def solve_07(input_size, hidden_size, num_layers):
    class DecoderRNN(nn.Module):
        def __init__(self,input_size,hidden_size,num_layers):
            # 초기화 전체를 완성하세요. super().__init__()도 포함합니다.
            raise NotImplementedError('???? 초기화')
        def forward(self,x, h):
            raise NotImplementedError('???? forward')
    return DecoderRNN(input_size,hidden_size,num_layers)

In [ ]:
# 이 검증 셀은 수정하지 마세요.
def check_07():
    expected=(((torch.tensor([[[-0.6058937311172485], [-0.6283125877380371], [-0.5625663995742798], [-0.6042851209640503], [-0.572126030921936]], [[-0.691118597984314], [-0.5504676699638367], [-0.606425404548645], [-0.59322589635849], [-0.5658470988273621]]],dtype=torch.float32), torch.tensor([[[-0.47861382365226746, -0.09292124211788177, -0.3890649974346161], [-0.4197760820388794, -0.17723697423934937, -0.36758291721343994]]],dtype=torch.float32),), (('rnn.weight_ih_l0', torch.tensor([[0.005369961261749268], [0.030882172286510468], [0.10490234196186066]],dtype=torch.float32),), ('rnn.weight_hh_l0', torch.tensor([[0.1594219207763672, -0.07635293900966644, 0.12381640076637268], [0.08113405853509903, -0.040205225348472595, 0.07894109934568405], [0.24973323941230774, -0.08177858591079712, 0.21773876249790192]],dtype=torch.float32),), ('rnn.bias_ih_l0', torch.tensor([-0.299585223197937, -0.20653608441352844, -0.523729681968689],dtype=torch.float32),), ('rnn.bias_hh_l0', torch.tensor([-0.2995852530002594, -0.20653608441352844, -0.5237296223640442],dtype=torch.float32),), ('fc.weight', torch.tensor([[0.5036031007766724, 0.024070434272289276, 0.6140089631080627]],dtype=torch.float32),), ('fc.bias', torch.tensor([-1.1960536241531372],dtype=torch.float32),),),), ((torch.tensor([[[0.23438048362731934, 0.7169764041900635], [0.12744539976119995, 0.7060267925262451], [0.15848708152770996, 0.7460330724716187], [0.4182033836841583, 0.7243571877479553], [0.27434176206588745, 0.7518713474273682]], [[0.7450439929962158, 0.6352075934410095], [0.32358211278915405, 0.7685457468032837], [0.19270870089530945, 0.7689639329910278], [0.21368935704231262, 0.7459865808486938], [0.1465800404548645, 0.7509602308273315]]],dtype=torch.float32), torch.tensor([[[-0.6083542704582214, -0.046665605157613754, 0.4021404981613159, 0.4690534174442291], [-0.3539528548717499, -0.3176889419555664, 0.6384735107421875, 0.07450108230113983]], [[0.18021252751350403, -0.7198556661605835, 0.6443697810173035, 0.5545638799667358], [-0.10383477061986923, -0.7720419764518738, 0.7811237573623657, 0.5141549706459045]]],dtype=torch.float32),), (('rnn.weight_ih_l0', torch.tensor([[0.029521053656935692, 0.02677426114678383], [-0.0191130880266428, -0.04845613241195679], [0.010008063167333603, 0.06010524556040764], [-0.02575644664466381, -0.03684394434094429]],dtype=torch.float32),), ('rnn.weight_hh_l0', torch.tensor([[0.006545810028910637, -0.015128605999052525, -0.03514067828655243, 0.00035702879540622234], [0.011251053772866726, -0.010264020413160324, -0.03307877108454704, -0.00281181326135993], [-0.0560036338865757, 0.03358372673392296, 0.01197768747806549, 0.009778755716979504], [-0.0690922662615776, 0.026773246005177498, 0.021953536197543144, 0.021607723087072372]],dtype=torch.float32),), ('rnn.bias_ih_l0', torch.tensor([-0.03684543818235397, -0.020137451589107513, 0.08482527732849121, 0.12359412014484406],dtype=torch.float32),), ('rnn.bias_hh_l0', torch.tensor([-0.03684543818235397, -0.020137455314397812, 0.08482528477907181, 0.12359412014484406],dtype=torch.float32),), ('rnn.weight_ih_l1', torch.tensor([[-0.08043690025806427, 0.005201973021030426, 0.06998394429683685, 0.0415613055229187], [0.17061246931552887, -0.024528954178094864, -0.14912894368171692, -0.07063832134008408], [-0.041102781891822815, -0.02555166184902191, 0.031546853482723236, 0.013116740621626377], [-0.12339741736650467, -0.016998890787363052, 0.12411411106586456, 0.05209522694349289]],dtype=torch.float32),), ('rnn.weight_hh_l1', torch.tensor([[-0.0030397847294807434, -0.09597048908472061, 0.08255261927843094, 0.09215236455202103], [0.010925203561782837, 0.23309797048568726, -0.22318574786186218, -0.19613665342330933], [0.057161349803209305, -0.06219800189137459, 0.07784851640462875, 0.03663475811481476], [0.020160341635346413, -0.16569465398788452, 0.155620276927948, 0.15099908411502838]],dtype=torch.float32),), ('rnn.bias_ih_l1', torch.tensor([0.15585295855998993, -0.34891045093536377, 0.07534971833229065, 0.2575584053993225],dtype=torch.float32),), ('rnn.bias_hh_l1', torch.tensor([0.15585295855998993, -0.3489104211330414, 0.07534971833229065, 0.2575584352016449],dtype=torch.float32),), ('fc.weight', torch.tensor([[0.08856527507305145, -0.19503799080848694, 0.12564462423324585, 0.17448681592941284], [0.11011836677789688, -0.5029958486557007, 0.4419969320297241, 0.4279686510562897]],dtype=torch.float32),), ('fc.bias', torch.tensor([0.28344622254371643, 0.7314929366111755],dtype=torch.float32),),),),)
    for case in range(2):
        torch.manual_seed(3070+case)
        input_size=case+1
        hidden_size=3+case
        num_layers=case+1
        actual=solve_07(input_size, hidden_size, num_layers)
        same(observe(actual,'DecoderRNN',case,locals()),expected[case])
    print('PASS · 07 · 두 입력 검증')
try:
    check_07()
except Exception as error:
    print('FAIL · 07 ·',type(error).__name__,str(error))


<details><summary>해설 · 공식 · 정답 보기 (먼저 풀어보세요)</summary>

Decoder는 입력x와 전달받은 h를 RNN에 넣고 output을 FC로 투영합니다. 반환(out,h). solution의 FC(H,input_size)를 기준으로 다변량도 검사합니다. 정답 파일의 input_size를 그대로 사용합니다.

```python
def solve_07(input_size, hidden_size, num_layers):
    class DecoderRNN(nn.Module):
      def __init__(self, input_size, hidden_size, num_layers):
        super(DecoderRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, input_size)

      def forward(self, x, h):
        out, h = self.rnn(x, h)
        out = self.fc(out)
        return out, h
    return DecoderRNN(input_size,hidden_size,num_layers)
```

테스트용 함수 인자·반환값을 추가하고 긴 실행·다운로드·그래프는 제외했습니다.
</details>